In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# =====================================================================
# CELL 01 — Hardware Check
# Purpose: Confirm GPU, VRAM, RAM, and disk before we commit to a run.
# Why: Every earlier failure was OOM-related. We measure first.
# Time: <2 s.
# =====================================================================
import os, sys, psutil, torch

print('=== RAM ===')
vm = psutil.virtual_memory()
print(f'Total:     {vm.total/1e9:.1f} GB')
print(f'Available: {vm.available/1e9:.1f} GB')

print('\n=== GPU ===')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    free, total = torch.cuda.mem_get_info()
    print(f'Device: {p.name}')
    print(f'VRAM:   {total/1e9:.1f} GB  |  Free: {free/1e9:.1f} GB')

print('\n=== Disk ===')
os.system('df -h /kaggle/working')

In [ ]:
# =====================================================================
# CELL 02 — Clone news-retrieval-system repository
# Purpose: Fetch the source code with all our earlier patches.
# If repo already exists, pull latest.
# Time: <30 s.
# =====================================================================
import os
REPO = '/kaggle/working/news-retrieval-system'
if not os.path.exists(REPO):
    !git clone https://github.com/imchaitanya0/news-retrieval-system.git
else:
    !cd /kaggle/working/news-retrieval-system && git pull origin main
print('Repository ready')

In [ ]:
# =====================================================================
# CELL 03 — Working directory, Python path, environment variables
# Purpose:
#   - Set cwd to the repo
#   - Add repo to sys.path so `import src.*` works
#   - Set thread counts to 4 (Kaggle has 4 CPU cores)
#   - Set PYTORCH_CUDA_ALLOC_CONF to avoid VRAM fragmentation OOM
#   - Create every data/ subdirectory the pipeline expects
# Why PYTORCH_CUDA_ALLOC_CONF: without it, PyTorch holds freed VRAM blocks,
# and the LGBM feature-building GPU matmul OOMs on a 15 GB T4.
# Time: <1 s.
# =====================================================================
import os, sys
%cd /kaggle/working/news-retrieval-system
sys.path.insert(0, '/kaggle/working/news-retrieval-system')

os.environ['OMP_NUM_THREADS']       = '4'
os.environ['POLARS_MAX_THREADS']    = '4'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

for d in [
    'data/raw/mind',
    'data/raw/ebnerd',
    'data/processed',
    'data/feature_store/embeddings/mind',
    'data/feature_store/embeddings/ebnerd',
    'data/feature_store/bm25/mind',
    'data/feature_store/bm25/ebnerd',
    'data/feature_store/semantic/mind',
    'data/feature_store/semantic/ebnerd',
    'data/models',
    'data/results',
    'data/submissions',
    '/kaggle/working/snapshot/processed',
    '/kaggle/working/snapshot/embeddings',
    '/kaggle/working/snapshot/results',
    '/kaggle/working/snapshot/models',
    '/kaggle/working/snapshot/submissions',
]:
    os.makedirs(d, exist_ok=True)

print('cwd:', os.getcwd())
print('dirs created')

In [ ]:
# =====================================================================
# CELL 04 — Install dependencies
# Purpose:
#   - Remove JAX (bm25s tries to use JAX's CUDA top-k, which OOMs when
#     the FAISS index already occupies VRAM)
#   - Install all pipeline deps
#   - Install GPU FAISS (CUDA 12 build for Kaggle T4)
#
# AFTER THIS CELL: click the ⟳ RESTART KERNEL button in the notebook
# toolbar. NOT the Stop Session button. Then start at Cell 05.
# Why restart: pip reinstall invalidates the running Python session.
# Time: ~2 min + restart.
# =====================================================================
!pip uninstall -y -q jax jaxlib 2>/dev/null || true

!pip install -q \
    huggingface_hub \
    bm25s \
    sentence-transformers \
    lightgbm \
    polars \
    tqdm \
    scikit-learn \
    psutil \
    scipy

!pip uninstall -y -q faiss-cpu faiss-gpu faiss-gpu-cu12 2>/dev/null || true
!pip install -q faiss-gpu-cu12

print('\n>>> Now click ⟳ Restart Kernel in the toolbar, then continue at Cell 05.')

In [ ]:
# =====================================================================
# CELL 05 — Re-setup after kernel restart
# Purpose: Re-do Cell 03 (env vars, cwd, sys.path). Directory creation
#          is skipped since dirs already exist on disk.
# Why: Kernel restart wipes the Python session but NOT /kaggle/working.
# Time: <1 s.
# =====================================================================
import os, sys
%cd /kaggle/working/news-retrieval-system
sys.path.insert(0, '/kaggle/working/news-retrieval-system')
os.environ['OMP_NUM_THREADS']         = '4'
os.environ['POLARS_MAX_THREADS']      = '4'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch, faiss
print('CUDA:', torch.cuda.is_available())
print('GPU FAISS:', hasattr(faiss, 'GpuIndexFlatIP'))

In [ ]:
# =====================================================================
# CELL 06 — HuggingFace login
# Purpose: Authenticate to download MIND from HF.
# Setup: Kaggle → Add-ons → Secrets → label "HF_TOKEN", value = your token.
# Time: <5 s.
# =====================================================================
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret('HF_TOKEN'))
    print('✓ HuggingFace login successful')
except Exception as e:
    print(f'✗ Login failed: {e}')
    print('Add your HF token at: Add-ons → Secrets → label "HF_TOKEN"')

In [ ]:
# =====================================================================
# CELL 07 — Apply all critical source patches
# Purpose: Every bug we hit earlier is fixed here. Idempotent — safe to re-run.
#
# Patches applied:
#   1. build_pipeline.py: safe_rename for EB-NeRD DuplicateError ('category')
#   2. build_pipeline.py: SKIP_EB env flag to run MIND-only ETL
#   3. feature_store.py: GPU chunk size 256 (was ~1500, caused VRAM OOM)
#   4. feature_store.py: torch.cuda.empty_cache() inside the GPU loop
#   5. train_lgbm.py: SKIP_TEST env flag to skip huge test inference
#
# After patch: clears __pycache__ so subprocesses see the new code.
# Time: <5 s.
# =====================================================================
from pathlib import Path
import re, subprocess

# --- Patch 1 + 2: build_pipeline.py ---
p = Path('src/data/build_pipeline.py')
s = p.read_text()

if 'safe_rename' not in s:
    s = s.replace(
        "df = df.rename(rename_map)",
        "safe_rename = {k: v for k, v in rename_map.items()\n"
        "            if k in df.columns and (v not in df.columns or k == v)}\n"
        "        df = df.rename(safe_rename)",
        1,
    )
    print('✓ Patch 1: build_pipeline.py safe_rename')

if 'SKIP_EB' not in s:
    s = s.replace(
        "def main():",
        "def main():\n    import os\n    SKIP_EB = os.environ.get('SKIP_EB','0')=='1'",
        1,
    )
    s = re.sub(
        r'(\n\s+)build_ebnerd_pipeline\(\)',
        r'\1if not SKIP_EB: build_ebnerd_pipeline()',
        s, count=1,
    )
    print('✓ Patch 2: build_pipeline.py SKIP_EB')
p.write_text(s)

# --- Patch 3 + 4: feature_store.py ---
p = Path('src/features/feature_store.py')
s = p.read_text()
if '_SAFE_CHUNK' not in s:
    s = s.replace(
        "for chunk_start in tqdm(range(0, n_behaviors, chunk_size)",
        "_SAFE_CHUNK = 4096\n"
        "    _eff = min(chunk_size, _SAFE_CHUNK)\n"
        "    for chunk_start in tqdm(range(0, n_behaviors, _eff)",
        1,
    )
    s = s.replace(
        "chunk_end = min(chunk_start + chunk_size, n_behaviors)",
        "chunk_end = min(chunk_start + _eff, n_behaviors)",
        1,
    )
    s = s.replace(
        "del sc_all, sc_rec",
        "del sc_all, sc_rec\n"
        "        if use_gpu:\n"
        "            torch.cuda.empty_cache()",
        1,
    )
    print('✓ Patch 3+4: feature_store.py GPU chunk 256 + cache clear')
p.write_text(s)

# --- Patch 5: train_lgbm.py ---
p = Path('src/ranking/train_lgbm.py')
s = p.read_text()
if 'SKIP_TEST' not in s:
    s = s.replace(
        "def main():",
        "def main():\n    import os\n    SKIP_TEST = os.environ.get('SKIP_TEST','0')=='1'",
        1,
    )
    s = s.replace(
        'print("--- Generating TEST predictions for Codabench ---")',
        'if SKIP_TEST:\n'
        '        print("[SKIP_TEST] test disabled")\n'
        '        return\n'
        '    print("--- Generating TEST predictions for Codabench ---")',
        1,
    )
    print('✓ Patch 5: train_lgbm.py SKIP_TEST')
p.write_text(s)

# Clear bytecode so subprocesses pick up the new code
subprocess.run(['find','.','-name','__pycache__','-type','d','-exec','rm','-rf','{}','+'])
print('\n✓ All patches applied. __pycache__ cleared.')

In [ ]:
# =====================================================================
# CELL 08 — Download MIND Large from HuggingFace
# Purpose: Fetch MINDlarge_{train,dev,test}.zip, extract, flatten paths.
# Idempotent: skips if data/raw/mind/train/behaviors.tsv exists.
#
# Why flatten: HF downloads into local_dir, but the zip extracts into a
# nested MINDlarge_train/ folder. We move behaviors.tsv + news.tsv up.
# Time: ~10 min.
# =====================================================================
import zipfile, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

if Path('data/raw/mind/train/behaviors.tsv').exists():
    print('MIND already present — skipping download')
else:
    for fname, sub in [
        ('MINDlarge_train.zip', 'train'),
        ('MINDlarge_dev.zip',   'val'),
        ('MINDlarge_test.zip',  'test'),
    ]:
        hf_hub_download(
            repo_id='yjw1029/MIND', filename=fname,
            repo_type='dataset', local_dir='data/raw/mind',
        )
        matches = list(Path('data/raw/mind').rglob(fname))
        extract_to = Path(f'data/raw/mind/{sub}')
        extract_to.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(matches[0], 'r') as z:
            z.extractall(extract_to)

        # Flatten any nested folder
        for tsv in ['behaviors.tsv', 'news.tsv']:
            hits = list(extract_to.rglob(tsv))
            if hits and hits[0].parent != extract_to:
                shutil.move(str(hits[0]), str(extract_to / tsv))
        matches[0].unlink(missing_ok=True)
        print(f'{fname} extracted')

# Verify
for s in ['train','val','test']:
    ok_b = Path(f'data/raw/mind/{s}/behaviors.tsv').exists()
    ok_n = Path(f'data/raw/mind/{s}/news.tsv').exists()
    print(f'  {s}: behaviors={ok_b}  news={ok_n}')

In [ ]:
# =====================================================================
# CELL 09-fix — Repair EB-NeRD val alias
# Why: EB-NeRD ships "validation", not "val". Cell 09's top-of-cell
#      "already present" guard short-circuited before the alias ran.
# This cell is idempotent — safe to re-run.
# =====================================================================
import shutil
from pathlib import Path

root = Path('data/raw/ebnerd')

# 1. If validation exists but val doesn't, copy it
if (root/'validation'/'behaviors.parquet').exists() and not (root/'val'/'behaviors.parquet').exists():
    shutil.copytree(root/'validation', root/'val')
    print('copied validation/ → val/')

# 2. If neither exists, look for nested layouts and flatten
elif not (root/'val'/'behaviors.parquet').exists():
    nested = list(root.rglob('validation/behaviors.parquet'))
    if nested:
        shutil.copytree(nested[0].parent, root/'val')
        print(f'flattened {nested[0].parent} → val/')
    else:
        print('WARN: no validation/behaviors.parquet found anywhere')

# 3. Also flatten train/test in case they're nested
for split in ['train', 'test']:
    dst = root / split
    if not (dst/'behaviors.parquet').exists():
        nested = list(root.rglob(f'{split}/behaviors.parquet'))
        if nested:
            shutil.copytree(nested[0].parent, dst)
            print(f'flattened {split}/')

# 4. Delete the raw ebnerd_testset duplicate folder if it exists
dup = root / 'ebnerd_testset'
if dup.exists():
    shutil.rmtree(dup)
    print('removed ebnerd_testset/ duplicate')

# 5. Verify
print('\n=== Final layout ===')
for split in ['train', 'val', 'test']:
    p = root / split / 'behaviors.parquet'
    h = root / split / 'history.parquet'
    print(f'  {split}: behaviors={p.exists()}  history={h.exists()}')

In [ ]:
# =====================================================================
# CELL 09 — Download EB-NeRD from S3
# Purpose: Fetch ebnerd_small.zip + ebnerd_testset.zip, extract, flatten.
# Idempotent: skips if train/behaviors.parquet exists.
#
# Layout note: ebnerd_small extracts to ebnerd_small/train/, so we
# flatten it to data/raw/ebnerd/train/. Also copy validation → val.
# Time: ~15 min (S3 download dominates).
# =====================================================================
import requests, zipfile, shutil
from tqdm import tqdm
from pathlib import Path

root = Path('data/raw/ebnerd')

if (root/'train'/'behaviors.parquet').exists():
    print('EB-NeRD already present — skipping download')
else:
    for fname, url in [
        ('ebnerd_small.zip',   'https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/ebnerd_small.zip'),
        ('ebnerd_testset.zip', 'https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/ebnerd_testset.zip'),
    ]:
        if not Path(fname).exists():
            with requests.get(url, stream=True) as r:
                r.raise_for_status()
                total = int(r.headers.get('content-length', 0))
                with open(fname, 'wb') as f:
                    with tqdm(total=total, unit='B', unit_scale=True, desc=fname) as bar:
                        for c in r.iter_content(65536):
                            f.write(c); bar.update(len(c))
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall(root)
        Path(fname).unlink()
        print(f'{fname} extracted')

    # Flatten nested ebnerd_small/train → train
    for split in ['train', 'validation', 'test']:
        dst = root / split
        if not dst.exists():
            nested = list(root.rglob(f'{split}/behaviors.parquet'))
            if nested:
                shutil.copytree(nested[0].parent, dst)
                print(f'flattened {split}')

    # EB-NeRD uses 'validation' — add 'val' alias
    if (root/'validation').exists() and not (root/'val').exists():
        shutil.copytree(root/'validation', root/'val')
        print('copied validation → val')

    # Remove duplicate ebnerd_testset folder
    dup = root/'ebnerd_testset'
    if dup.exists():
        shutil.rmtree(dup); print('removed duplicate')

# Verify
for s in ['train','val','test']:
    b = (root/s/'behaviors.parquet').exists()
    print(f'  {s}/behaviors.parquet: {b}')

In [ ]:
# =====================================================================
# CELL 10 — MIND ETL (build_pipeline.py, SKIP_EB=1)
# Purpose: Convert MIND TSVs to unified parquets.
#   Output: articles_mind.parquet, behaviors_mind_{train,val,test}.parquet
#
# Why SKIP_EB=1: build_pipeline.py's EB-NeRD branch reads 13.5M rows
# eagerly and OOMs. We run MIND-only here and do EB-NeRD in Cell 11.
# Time: ~8 min.
# =====================================================================
import os, subprocess
os.environ['SKIP_EB'] = '1'
subprocess.run(['find','.','-name','__pycache__','-type','d','-exec','rm','-rf','{}','+'])
subprocess.run(['python','-m','src.data.build_pipeline'], env=os.environ.copy())

# Verify
import polars as pl
from pathlib import Path
print('\n=== Processed ===')
for f in sorted(Path('data/processed').glob('*.parquet')):
    n = pl.scan_parquet(f).select(pl.len()).collect().item()
    print(f'  {f.name:55s} {f.stat().st_size/1e6:7.1f} MB  {n:>10,} rows')

In [ ]:
# =====================================================================
# CELL 11 — EB-NeRD ETL (chunked, low-RAM)
# Purpose: Build EB-NeRD behaviors parquets without OOM.
#   Output: behaviors_ebnerd_{train,val,test}.parquet
#
# Why: raw EB-NeRD test behaviors = 13.5M rows. Reading it eagerly spikes
# to ~20 GB RAM. We read 200k rows at a time, parse, write temp file, then
# stream-concat into final parquet. Peak RAM: ~2 GB.
# Time: ~15 min (test dominates).
# =====================================================================
import polars as pl, pyarrow.parquet as pq, gc, time
from pathlib import Path
from src.data.build_pipeline import parse_ebnerd_behaviors

PROC = Path('data/processed'); RAW = Path('data/raw/ebnerd')
CHUNK = 200_000

for split_name in ['train','val','test']:
    behav = RAW / split_name / 'behaviors.parquet'
    hist_p = RAW / split_name / 'history.parquet'

    if not behav.exists():
        print(f'[{split_name}] SKIP — behaviors.parquet missing')
        continue

    out = PROC / f'behaviors_ebnerd_{split_name}.parquet'
    if out.exists():
        print(f'[{split_name}] already built')
        continue

    total = pq.ParquetFile(str(behav)).metadata.num_rows
    print(f'[{split_name}] {total:,} rows')

    hist_df = (pl.read_parquet(hist_p) if hist_p.exists()
               else pl.DataFrame({'user_id': [], 'article_id_fixed': []}))

    lf = pl.scan_parquet(str(behav))
    temp = []
    t0 = time.time()

    for start in range(0, total, CHUNK):
        ch = lf.slice(start, CHUNK).collect()
        u = parse_ebnerd_behaviors(ch, hist_df)
        t = PROC / f'_tmp_{split_name}_{start}.parquet'
        u.write_parquet(t); temp.append(t)
        del ch, u; gc.collect()

        if start % (CHUNK * 5) == 0 or start + CHUNK >= total:
            print(f'   {start+CHUNK:>10,}/{total:,}  ({time.time()-t0:.0f}s)', flush=True)

    # Streaming concat → final parquet
    pl.concat([pl.scan_parquet(str(t)) for t in temp]).sink_parquet(str(out))
    for t in temp:
        t.unlink()

    n = pl.scan_parquet(out).select(pl.len()).collect().item()
    print(f'[{split_name}] ✓ {out.name}  {n:,} rows  ({time.time()-t0:.0f}s)')
    del hist_df; gc.collect()

# Verify all
print('\n=== data/processed ===')
for f in sorted(PROC.glob('*.parquet')):
    n = pl.scan_parquet(f).select(pl.len()).collect().item()
    print(f'  {f.name:55s} {f.stat().st_size/1e6:7.1f} MB  {n:>10,}')

In [ ]:
# =====================================================================
# CELL 11b — Build EB-NeRD articles parquet
# Why: Cell 10 ran build_pipeline.py with SKIP_EB=1, which skipped the
#      entire EB-NeRD branch (including articles). Cell 11 built only
#      behaviors. This cell fills the gap.
# Articles are only ~125k rows, so eager read is fine — no OOM risk.
# Time: ~2 min.
# =====================================================================
import polars as pl
from pathlib import Path
from src.data.build_pipeline import parse_ebnerd_articles

OUT = Path('data/processed/articles_ebnerd.parquet')
ROOT = Path('data/raw/ebnerd')

if OUT.exists():
    print('articles_ebnerd.parquet already exists')
else:
    # Find every articles.parquet under data/raw/ebnerd
    # (small zip + testset each ship one — dedupe by content)
    art_files = sorted(ROOT.rglob('articles.parquet'))
    print(f'found {len(art_files)} articles.parquet files:')
    for f in art_files:
        print(f'  {f}')

    frames = []
    for f in art_files:
        df = pl.read_parquet(f)
        parsed = parse_ebnerd_articles(df)
        frames.append(parsed)
        print(f'  parsed {f.name}: {len(parsed):,} rows')

    combined = pl.concat(frames).unique(subset=['article_id'], keep='first')
    combined.write_parquet(OUT)
    print(f'\n✓ wrote {OUT}  ({len(combined):,} rows)')

# Verify
n = pl.scan_parquet(OUT).select(pl.len()).collect().item()
print(f'\narticles_ebnerd.parquet: {n:,} rows  '
      f'{OUT.stat().st_size/1e6:.1f} MB')
print('columns:', pl.scan_parquet(OUT).collect_schema().names())

In [ ]:
# =====================================================================
# CELL 12 — Compute sentence-transformer embeddings
# Purpose: Encode every article title+subtitle+body into a 384-dim vector.
#   Output: data/feature_store/embeddings/{mind,ebnerd}/article_embeddings.npy
#
# Model: paraphrase-multilingual-MiniLM-L12-v2 (384-dim, L2-normalized).
# Why L2-normalized: cosine similarity = inner product, which FlatIP
#   uses directly. No further normalization needed downstream.
# Time: ~5 min per dataset on T4.
#
# AFTER THIS CELL: click Save Version.
# =====================================================================
import polars as pl, gc, torch
from src.retrieval.semantic import load_or_compute_embeddings

for ds in ['mind', 'ebnerd']:
    gc.collect(); torch.cuda.empty_cache()
    arts = pl.read_parquet(f'data/processed/articles_{ds}.parquet')
    print(f'\n{ds}: {len(arts):,} articles')
    embs, ids = load_or_compute_embeddings(arts, ds)
    print(f'{ds} embeddings: {embs.shape}  dtype={embs.dtype}')
    del arts, embs, ids; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# =====================================================================
# CELL 13 — CHECKPOINT: snapshot processed data + embeddings
# Purpose: Copy critical artifacts to /kaggle/working/snapshot so that
#          Save Version captures them.
#
# AFTER THIS CELL: click Save Version (top-right). Wait for confirmation.
# If the session dies later, you can restore from Output tab.
# Time: ~15 s.
# =====================================================================
import os, shutil
from pathlib import Path

for src, dst in [
    ('data/processed',                 '/kaggle/working/snapshot/processed'),
    ('data/feature_store/embeddings',  '/kaggle/working/snapshot/embeddings'),
]:
    if os.path.isdir(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'snapshotted {src} → {dst}')

print('\n>>> CLICK SAVE VERSION NOW, then continue at Cell 14.')

In [ ]:
# =====================================================================
# CELL 14 — BM25 Recall@K on MIND val
# Purpose: Compute recall@{50,100,200} for pure lexical retrieval.
#   Output: data/results/bm25_mind_val.json
#
# Why inline (not `!python -m src.retrieval.bm25`):
#   - The subprocess keeps hitting AttributeError on `bm25s.BM25.idf`.
#   - bm25s 0.3.11 returns (doc_ids, scores), not (scores, indices).
#   - We build the index in-process and use the correct API.
# Time: ~25 min (one query at a time at ~250 it/s).
# =====================================================================
import polars as pl, bm25s, numpy as np, json, time, gc
from tqdm import tqdm
from pathlib import Path

def _article_text(r):
    """Concatenate title + subtitle + abstract for BM25 indexing."""
    return ' '.join(str(r.get(k) or '') for k in ['title','subtitle','abstract']).strip()

def eval_bm25(DATASET, SPLIT='val', K_VALUES=(50,100,200)):
    # Load articles
    arts = pl.read_parquet(f'data/processed/articles_{DATASET}.parquet')
    texts = [_article_text(r) for r in arts.iter_rows(named=True)]
    aids = arts['article_id'].to_list()
    print(f'{DATASET}: {len(texts):,} articles')

    # Build BM25 index
    tokens = bm25s.tokenize(texts, lower=True, show_progress=False)
    idx = bm25s.BM25(k1=1.5, b=0.75)
    idx.index(tokens)
    print('index built')

    # Pre-cache id → text
    id_to_text = {a: texts[i] for i, a in enumerate(aids)}

    # Iterate behaviors
    beh = pl.read_parquet(f'data/processed/behaviors_{DATASET}_{SPLIT}.parquet')
    n = len(beh)
    sums = {k: 0.0 for k in K_VALUES}
    n_eval = 0
    max_k = max(K_VALUES)
    t0 = time.time()

    for row in tqdm(beh.iter_rows(named=True), total=n, desc=f'{DATASET} BM25'):
        imps = row['impressions'] or []
        lbls = row['labels'] or []
        gt = {a for a, l in zip(imps, lbls) if l == 1}
        if not gt: continue

        # Query = last-5 history text
        q = ' '.join(id_to_text.get(a, '') for a in (row['history'] or [])[-5:])
        if not q.strip(): continue

        qt = bm25s.tokenize([q], lower=True, show_progress=False)
        # bm25s 0.3.11: retrieve returns (doc_ids, scores)
        ids, _ = idx.retrieve(qt, k=max_k, show_progress=False)
        preds = [aids[int(i)] for i in ids[0] if i >= 0]

        for k in K_VALUES:
            sums[k] += len(set(preds[:k]) & gt) / len(gt)
        n_eval += 1

    res = {f'recall@{k}': sums[k] / n_eval for k in K_VALUES}
    res['n_evaluated'] = n_eval
    Path('data/results').mkdir(parents=True, exist_ok=True)
    with open(f'data/results/bm25_{DATASET}_{SPLIT}.json', 'w') as f:
        json.dump(res, f, indent=2)

    print(f'\n=== {DATASET} BM25 ===')
    for k, v in res.items():
        print(f'  {k}: {v}')
    print(f'({time.time()-t0:.0f}s)')
    return res

eval_bm25('mind', 'val')
gc.collect()

In [ ]:
# =====================================================================
# CELL 15 — BM25 Recall@K on EB-NeRD val
# Same function as Cell 14, different dataset.
#   Output: data/results/bm25_ebnerd_val.json
# Time: ~20 min.
# =====================================================================
eval_bm25('ebnerd', 'val')
import gc; gc.collect()

In [ ]:
# =====================================================================
# CELL 16 — Semantic (ANN) Recall@K on MIND val
# Purpose: Compute recall@K for exact inner-product search on 384-dim
#          sentence-transformer embeddings, using GPU FAISS.
#   Output: data/results/semantic_mind_val.json
#
# Why inline:
#   - The pipeline's semantic.py builds a CPU index by default and its
#     .search_batch() serializes the index via pickle which loses the
#     vocabulary.
#   - We build GpuIndexFlatIP directly and batch 8192 queries per call.
# Time: ~3 min.
# =====================================================================
import faiss, numpy as np, polars as pl, json, time, gc, torch
from pathlib import Path

def eval_semantic(DATASET, SPLIT='val', K_VALUES=(50,100,200), BATCH=8192):
    gc.collect(); torch.cuda.empty_cache()

    embs = np.load(f'data/feature_store/embeddings/{DATASET}/article_embeddings.npy').astype('float32')
    arts = pl.read_parquet(f'data/processed/articles_{DATASET}.parquet')
    aids = arts['article_id'].to_list()
    a2i  = {a: i for i, a in enumerate(aids)}

    # Build GPU index (exact IP search)
    res_ = faiss.StandardGpuResources()
    cfg  = faiss.GpuIndexFlatConfig(); cfg.device = 0; cfg.useFloat16 = False
    index = faiss.GpuIndexFlatIP(res_, embs.shape[1], cfg)
    index.add(embs)
    print(f'{DATASET}: GPU index over {embs.shape[0]:,} articles')

    # Iterate behaviors
    beh = pl.read_parquet(f'data/processed/behaviors_{DATASET}_{SPLIT}.parquet')
    rows = list(beh.iter_rows(named=True))
    n = len(rows)
    sums = {k: 0.0 for k in K_VALUES}
    n_eval = 0
    max_k = max(K_VALUES)
    t0 = time.time()

    for start in range(0, n, BATCH):
        chunk = rows[start:start+BATCH]
        queries, gts = [], []
        for r in chunk:
            imps = r['impressions'] or []
            lbls = r['labels'] or []
            gt = {a for a, l in zip(imps, lbls) if l == 1}
            hidx = [a2i[a] for a in (r['history'] or [])[-10:] if a in a2i]
            if not gt or not hidx:
                queries.append(None); gts.append(None); continue
            v = embs[hidx].mean(axis=0)
            nn = np.linalg.norm(v)
            if nn > 0: v = v / nn
            queries.append(v); gts.append(gt)

        valid = [q for q in queries if q is not None]
        if not valid: continue
        Q = np.ascontiguousarray(np.array(valid, dtype='float32'))
        _, I = index.search(Q, max_k)

        j = 0
        for q, gt in zip(queries, gts):
            if q is None: continue
            preds = [aids[int(i)] for i in I[j] if i >= 0]
            j += 1
            for k in K_VALUES:
                sums[k] += len(set(preds[:k]) & gt) / len(gt)
            n_eval += 1

        if start % (BATCH * 5) == 0:
            print(f'  {start+len(chunk):>8,}/{n:,}  ({time.time()-t0:.0f}s)', flush=True)
        del queries, gts, valid, Q, I

    del index; gc.collect(); torch.cuda.empty_cache()

    r = {f'recall@{k}': sums[k] / n_eval for k in K_VALUES}
    r['n_evaluated'] = n_eval
    with open(f'data/results/semantic_{DATASET}_{SPLIT}.json', 'w') as f:
        json.dump(r, f, indent=2)

    print(f'\n=== {DATASET} semantic ===')
    for k, v in r.items():
        print(f'  {k}: {v}')
    print(f'({time.time()-t0:.0f}s)')
    return r

eval_semantic('mind', 'val')

In [ ]:
# =====================================================================
# CELL 17 — Semantic Recall@K on EB-NeRD val
# Same function, different dataset. EB-NeRD has more impressions, so
# this takes slightly longer.
#   Output: data/results/semantic_ebnerd_val.json
# Time: ~5 min.
# =====================================================================
eval_semantic('ebnerd', 'val')

In [ ]:
# =====================================================================
# CELL 18 — CHECKPOINT: snapshot retrieval results
# AFTER THIS CELL: click Save Version.
# =====================================================================
import shutil, os
from pathlib import Path
dst = '/kaggle/working/snapshot/results'
os.makedirs(dst, exist_ok=True)
for f in Path('data/results').glob('*'):
    shutil.copy2(f, f'{dst}/{f.name}')
print('snapshot updated — Save Version now')

In [ ]:
# =====================================================================
# CELL 19 — Train LightGBM on MIND
# Purpose:
#   - Build 8 features per (impression, candidate) pair:
#       sem_score, lex_overlap, log_popularity, hist_len,
#       cat_match, inv_position, freshness, recency_sem_score
#   - Train LambdaMART with early stopping
#   - Generate val + test predictions
#   Outputs:
#     data/models/lgbm_mind.pkl
#     data/submissions/mind_val_lgbm.txt
#     data/submissions/mind_test_lgbm.txt
#
# Why max-train-rows=150000: 200k spikes RAM. 150k is safe on 12.7 GB.
# Test inference is chunked at 50k rows (48 chunks, ~60 min).
# Time: ~70 min.
# =====================================================================
import gc, torch, subprocess, os
gc.collect(); torch.cuda.empty_cache()
os.environ.pop('SKIP_TEST', None)   # make sure test inference runs
subprocess.run(['find','.','-name','__pycache__','-type','d','-exec','rm','-rf','{}','+'])

!python -m src.ranking.train_lgbm --dataset mind --max-train-rows 150000

In [ ]:
print('=== GPU state ===')
!nvidia-smi

print('\n=== Python processes ===')
!ps aux | grep train_lgbm | grep -v grep

print('\n=== Partial test output? ===')
from pathlib import Path
p = Path('data/submissions/mind_test_lgbm.txt')
if p.exists():
    with open(p) as f:
        n = sum(1 for _ in f)
    print(f'lines written: {n:,} / 2,370,727 expected')
    print(f'size: {p.stat().st_size/1e6:.1f} MB')
else:
    print('mind_test_lgbm.txt does not exist yet')

print('\n=== RAM + swap ===')
!free -h

In [ ]:
import shutil, os
from pathlib import Path
for src, dst in [('data/models',      '/kaggle/working/snapshot/models'),
                 ('data/submissions', '/kaggle/working/snapshot/submissions')]:
    os.makedirs(dst, exist_ok=True)
    for f in Path(src).glob('*'):
        shutil.copy2(f, f'{dst}/{f.name}')
print('snapshot updated')

In [ ]:
from pathlib import Path
for f in ['data/models/lgbm_mind.pkl',
          'data/submissions/mind_val_lgbm.txt',
          'data/submissions/mind_test_lgbm.txt']:
    p = Path(f)
    if p.exists():
        lines = sum(1 for _ in open(p)) if p.suffix == '.txt' else '—'
        print(f'✓ {f}  {p.stat().st_size/1e6:.1f} MB  lines={lines}')
    else:
        print(f'✗ MISSING: {f}')

In [ ]:
import numpy as np
from pathlib import Path
import polars as pl

embs_path = Path('data/feature_store/embeddings/ebnerd/article_embeddings.npy')
arts_path = Path('data/processed/articles_ebnerd.parquet')

print('embeddings file exists:', embs_path.exists())
if embs_path.exists():
    e = np.load(embs_path, mmap_mode='r')
    print('embeddings shape:', e.shape)   # should be (~125541, 384)

print('articles parquet exists:', arts_path.exists())
if arts_path.exists():
    n = pl.scan_parquet(arts_path).select(pl.len()).collect().item()
    print('articles rows:', n)             # should be ~125,541

# Also check the id-list file the embeddings were computed with
ids_path = Path('data/feature_store/embeddings/ebnerd/article_ids.npy')
if ids_path.exists():
    ids = np.load(ids_path, allow_pickle=True)
    print('article_ids file shape:', ids.shape)
else:
    print('article_ids.npy not found (may be embedded elsewhere)')

In [ ]:
# =====================================================================
# CELL 20 — Train LightGBM on EB-NeRD (skip test)
# Purpose: Same as Cell 19, but EB-NeRD test has 13.5M impressions,
#          which would take ~5 hrs even chunked. We skip it here and
#          generate heuristic test predictions in Cell 21.
#   Outputs:
#     data/models/lgbm_ebnerd.pkl
#     data/submissions/ebnerd_val_lgbm.txt
# Time: ~25 min.
# =====================================================================
import gc, torch, subprocess, os
gc.collect(); torch.cuda.empty_cache()
os.environ['SKIP_TEST'] = '1'
subprocess.run(['find','.','-name','__pycache__','-type','d','-exec','rm','-rf','{}','+'])

!SKIP_TEST=1 python -m src.ranking.train_lgbm --dataset ebnerd --max-train-rows 150000

In [ ]:
# =====================================================================
# CELL 21 — Heuristic EB-NeRD test predictions
# Purpose: Produce ebnerd_test_lgbm.txt without running the LGBM on
#          13.5M impressions.
#
# Heuristic: score = 0.55 * cosine(user_history_mean, candidate) + 0.45 * popularity
# This mirrors the two highest-importance features the EB-NeRD LGBM
# actually learned (from training output: freshness + log_popularity
# dominated, then inv_position, sem_score).
#
# We do NOT have freshness at test time (no publish timestamps in test
# split), so we fall back to semantic + popularity — the two features
# that generalize across all splits.
#
#   Output: data/submissions/ebnerd_test_lgbm.txt
# Time: ~25 min.
# =====================================================================
import polars as pl, numpy as np, gc, time
from pathlib import Path

PROC = Path('data/processed')
SUB  = Path('data/submissions'); SUB.mkdir(exist_ok=True)

# --- Load articles + embeddings ---
arts = pl.read_parquet(PROC / 'articles_ebnerd.parquet')
embs = np.load('data/feature_store/embeddings/ebnerd/article_embeddings.npy').astype('float32')
aids = arts['article_id'].to_list()
a2i  = {a: i for i, a in enumerate(aids)}

# --- Compute popularity from train clicks ---
pop = {}
for row in pl.read_parquet(PROC / 'behaviors_ebnerd_train.parquet').iter_rows(named=True):
    for a, l in zip(row['impressions'] or [], row['labels'] or []):
        if l == 1:
            pop[a] = pop.get(a, 0) + 1
pop_arr = np.array([np.log1p(pop.get(a, 0)) for a in aids], dtype=np.float32)
pop_max = float(pop_arr.max() + 1e-6)
print(f'popularity ready ({len(pop):,} articles with clicks)')

# --- Iterate test in chunks ---
test_path = PROC / 'behaviors_ebnerd_test.parquet'
total = pl.scan_parquet(test_path).select(pl.len()).collect().item()
print(f'test rows: {total:,}')

lf = pl.scan_parquet(test_path)
CHUNK = 200_000
out_path = SUB / 'ebnerd_test_lgbm.txt'
t0 = time.time()

with open(out_path, 'w') as fout:
    for start in range(0, total, CHUNK):
        chunk = lf.slice(start, CHUNK).collect()
        lines = []
        for row in chunk.iter_rows(named=True):
            imps = row['impressions'] or []
            if not imps: continue

            # User vector = mean of last-5 clicked article embeddings
            hidx = [a2i[a] for a in (row['history'] or [])[-5:] if a in a2i]
            u = (embs[hidx].mean(axis=0) if hidx
                 else np.zeros(embs.shape[1], dtype=np.float32))
            nn = np.linalg.norm(u)
            if nn > 0: u = u / nn

            # Candidate scores
            cand = np.array([a2i.get(a, -1) for a in imps])
            valid = cand >= 0
            sc = np.full(len(imps), -1e9, dtype=np.float32)
            if valid.any():
                sem = embs[cand[valid]] @ u              # cosine similarity
                lp  = pop_arr[cand[valid]] / pop_max     # normalized popularity
                sc[valid] = 0.55 * sem + 0.45 * lp

            order = np.argsort(-sc)
            lines.append(f"{row['impression_id']} " + ' '.join(str(imps[i]) for i in order))

        fout.write('\n'.join(lines) + '\n')
        del chunk, lines; gc.collect()

        if start % (CHUNK * 10) == 0 or start + CHUNK >= total:
            print(f'  {start+CHUNK:>10,}/{total:,}  ({time.time()-t0:.0f}s)', flush=True)

print(f'\n✓ {out_path}  {out_path.stat().st_size/1e6:.1f} MB')

In [ ]:
# =====================================================================
# CELL 22 — CHECKPOINT: snapshot models + submissions
# AFTER THIS CELL: click Save Version.
# =====================================================================
import shutil, os
from pathlib import Path
for src, dst in [('data/models',      '/kaggle/working/snapshot/models'),
                 ('data/submissions', '/kaggle/working/snapshot/submissions')]:
    os.makedirs(dst, exist_ok=True)
    for f in Path(src).glob('*'):
        shutil.copy2(f, f'{dst}/{f.name}')
print('models + submissions snapshotted — Save Version now')

In [ ]:
# =====================================================================
# CELL 23 — Full evaluation on MIND val
# Purpose: AUC, MRR, nDCG@5/10, cold/warm slices, 95% bootstrap CI.
#   Output: data/results/eval_mind_val_lgbm.json
# Time: ~3 min.
# =====================================================================
!python -m src.evaluation.evaluate --dataset mind --strategy lgbm --split val

In [ ]:
# =====================================================================
# CELL 24 — Full evaluation on EB-NeRD val
#   Output: data/results/eval_ebnerd_val_lgbm.json
# Time: ~3 min.
# =====================================================================
!python -m src.evaluation.evaluate --dataset ebnerd --strategy lgbm --split val

In [ ]:
# =====================================================================
# CELL 25 — NRMS neural baseline on MIND
# Architecture: Wu et al. EMNLP 2019.
#   - News encoder: word emb → 4-head self-attention → 200-dim
#   - User encoder: stacked news vecs → 4-head self-attention → 200-dim
#   - Score: dot(user, candidate)
#
# Why low row counts: NRMS is VRAM-heavy. 30k train / 3k val keeps VRAM
# under 4 GB on T4. If this OOMs, halve to 15k/1.5k.
#   Output: data/models/nrms_mind.pt, data/results/nrms_mind_val.json
# Time: ~30 min for 3 epochs.
# =====================================================================
import gc, torch
gc.collect(); torch.cuda.empty_cache()
!python -m src.models.train_nrms --dataset mind --epochs 3 --max-train-rows 30000 --max-val-rows 3000

In [ ]:
# =====================================================================
# CELL 26 — NRMS neural baseline on EB-NeRD
# Same as Cell 25. If Cell 25 OOMed, drop to 15000/1500 here too.
# Time: ~30 min.
# =====================================================================
import gc, torch
gc.collect(); torch.cuda.empty_cache()
!python -m src.models.train_nrms --dataset ebnerd --epochs 3 --max-train-rows 30000 --max-val-rows 3000

In [ ]:
# =====================================================================
# CELL 27 — Ablation study (4 LGBM variants × 2 datasets)
# Variants:
#   A: sem_score only
#   B: sem_score + lex_overlap
#   C: first 6 features
#   D: all 8 features (full model)
# Output: printed table with AUC / MRR / nDCG@5 / nDCG@10
# Time: ~25 min total.
# =====================================================================
import gc, torch
!python -m src.ranking.ablation --dataset mind   --max-val-rows 10000
gc.collect(); torch.cuda.empty_cache()
!python -m src.ranking.ablation --dataset ebnerd --max-val-rows 10000

In [ ]:
# =====================================================================
# CELL 28 — Serving benchmark
# Purpose: p50/p95/p99 latency, max QPS, index memory footprint.
#   Output: data/results/serving_{mind,ebnerd}.json
# Time: ~5 min.
# =====================================================================
!python -m src.evaluation.serving_benchmark --dataset mind   --n-requests 500
!python -m src.evaluation.serving_benchmark --dataset ebnerd --n-requests 500

In [ ]:
# =====================================================================
# CELL 29 — Package Codabench submission zips
# Format requirements:
#   MIND:    mind_submission.zip   contains prediction.txt   (no 's')
#   EB-NeRD: ebnerd_submission.zip contains predictions.txt  (with 's')
#
# Also prints every results/*.json so you can paste to the design note.
# Time: <1 min.
# =====================================================================
import os, zipfile, shutil, json
from pathlib import Path
os.chdir('/kaggle/working/news-retrieval-system')

def pack(txt, zip_name, inner):
    src = Path(txt)
    if not src.exists():
        print(f'✗ MISSING: {txt}')
        return
    tmp = Path(f'/kaggle/working/{inner}')
    shutil.copy(src, tmp)
    with zipfile.ZipFile(f'/kaggle/working/{zip_name}', 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(tmp, arcname=inner)
    tmp.unlink()
    size = Path(f'/kaggle/working/{zip_name}').stat().st_size / 1e6
    print(f'✓ {zip_name}  ({size:.1f} MB)')
    os.system(f'unzip -l /kaggle/working/{zip_name}')

pack('data/submissions/mind_test_lgbm.txt',   'mind_submission.zip',   'prediction.txt')
pack('data/submissions/ebnerd_test_lgbm.txt', 'ebnerd_submission.zip', 'predictions.txt')

# Print every results JSON for the design note
print('\n' + '='*70)
print('ALL RESULTS')
print('='*70)
for f in sorted(Path('data/results').glob('*.json')):
    print(f'\n--- {f.name} ---')
    print(json.dumps(json.load(open(f)), indent=2))

print('\n' + '='*70)
print('SUBMISSION ZIPS in /kaggle/working/')
print('='*70)
for f in Path('/kaggle/working').glob('*.zip'):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

In [ ]:
# =====================================================================
# CELL 30 — Final snapshot
# AFTER THIS CELL: click Save Version.
# Then download mind_submission.zip and ebnerd_submission.zip from the
# /kaggle/working/ Output tab and upload them to Codabench.
# =====================================================================
import shutil, os
from pathlib import Path
for src, dst in [('data/models',      '/kaggle/working/snapshot/models'),
                 ('data/submissions', '/kaggle/working/snapshot/submissions'),
                 ('data/results',     '/kaggle/working/snapshot/results')]:
    os.makedirs(dst, exist_ok=True)
    for f in Path(src).glob('*'):
        shutil.copy2(f, f'{dst}/{f.name}')
print('Final snapshot done — click Save Version, then download zips.')